# Story B.1 — Basket Rules: Evaluation & Interpretation

Nguyen Sy Hung, Feb 2026

This notebook is a **read-only evaluation layer** for Story B.1 outputs produced by `B_basket_rules.ipynb`.

It focuses on:
- Rule-quality diagnostics (support/confidence/lift distributions)
- Popularity-bias checks using `token_stats.parquet`
- Rule coverage and structural summaries
- Interpretation helpers (decode + browse rule neighborhoods)
- Optional held-out evaluation (hit-rate@K / MRR) for **movie-only** runs using `interactions_test.parquet` (no leakage back into mining).

## Where the mining outputs live

New (current) layout (multiple runs per preset):
- `data/story-module-outputs/story_b/preset_<PRESET>_<TOKEN_FAMILY_MODE>_<TRANSACTIONS_INPUT>/<RUN_ID>/`
  - `tables/association_rules.parquet`
  - `tables/rules_human_readable.parquet` (optional)
  - `tables/frequent_itemsets.parquet` (optional)
  - `reports/run_manifest.json`
  - `reports/summary.md`

This notebook can also fall back to legacy layouts like `data/story-module-outputs/story_B_*` if present.

Outputs (optional): writes `eval_metrics.json` + `eval_summary.md` into **each** Story B.1 run’s `reports/` folder when `WRITE_EVAL_ARTIFACTS=True`.

In [ ]:
from __future__ import annotations

import json
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# --- Config (edit these if needed) ---
# Evaluate multiple Story B.1 runs under data/story-module-outputs/
#
# Current layout (recommended):
#   data/story-module-outputs/story_b/preset_<...>/<RUN_ID>/tables/association_rules.parquet
#
# Controls:
# - Set to "LATEST_PER_PRESET" (default) to evaluate the most recent RUN_ID under each preset_* folder
# - Set to "ALL" to evaluate every discovered run
# - Or set to a list of specific run identifiers from AVAILABLE_RUNS, e.g.:
#     ["story_b/preset_A_movie_only_reduced/20260226_151835"]
STORY_B_RUN_FOLDERS: str | list[str] | None = "LATEST_PER_PRESET"

# Choose which run is used for the deep-dive plots + interpretation cells below.
# If None, the notebook will pick a sensible default (first non-empty run if available).
ACTIVE_RUN: str | None = None
ACTIVE_RUN_WAS_SET = ACTIVE_RUN is not None

WRITE_EVAL_ARTIFACTS = True

# Held-out evaluation settings (only used when rules are movie-only)
LIKE_RATING_THRESHOLD = 4.0
MAX_EVAL_USERS = 20000  # keep modest for notebook runtime
K_LIST = [5, 10, 20]

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "story_modules").exists():
            return p
    raise RuntimeError(f"Could not locate repo root from: {start}")

REPO_ROOT = find_repo_root()
TABLE_DIR = REPO_ROOT / "data" / "preprocessed-data" / "tables"
OUTPUTS_ROOT = REPO_ROOT / "data" / "story-module-outputs"
STORY_B_ROOT = OUTPUTS_ROOT / "story_b"

def _has_story_b_tables(run_root: Path) -> bool:
    return (run_root / "tables" / "association_rules.parquet").exists()

def _list_story_b_runs_new_layout(outputs_root: Path) -> list[str]:
    """Return run identifiers relative to OUTPUTS_ROOT, like: story_b/preset_.../<RUN_ID>"""
    story_b_root = outputs_root / "story_b"
    if not story_b_root.exists():
        return []
    runs: list[str] = []
    for preset_dir in sorted(story_b_root.glob("preset_*")):
        if not preset_dir.is_dir():
            continue
        for run_dir in sorted(preset_dir.iterdir()):
            if not run_dir.is_dir():
                continue
            if _has_story_b_tables(run_dir):
                rel = run_dir.relative_to(outputs_root).as_posix()
                runs.append(rel)
    return runs

def _list_story_b_runs_legacy(outputs_root: Path) -> list[str]:
    """Legacy fallback: folders like data/story-module-outputs/story_B_* (no RUN_ID subdirs)."""
    runs: list[str] = []
    for p in sorted(outputs_root.glob("story_B_*")):
        if not p.is_dir():
            continue
        if _has_story_b_tables(p):
            runs.append(p.name)
    return runs

def list_story_b_runs(outputs_root: Path) -> list[str]:
    runs = _list_story_b_runs_new_layout(outputs_root)
    if runs:
        return runs
    return _list_story_b_runs_legacy(outputs_root)

def pick_latest_per_preset(run_ids: list[str]) -> list[str]:
    """For new-layout run ids (story_b/preset_.../<RUN_ID>), pick the max RUN_ID per preset folder."""
    latest: dict[str, str] = {}
    for rid in run_ids:
        parts = rid.split("/")
        if len(parts) == 3 and parts[0] == "story_b" and parts[1].startswith("preset_"):
            preset = parts[1]
            run_id = parts[2]
            key = preset
            prev = latest.get(key)
            if prev is None or run_id > prev:
                latest[key] = run_id
    if not latest:
        return run_ids
    picked = [f"story_b/{preset}/{run_id}" for preset, run_id in sorted(latest.items())]
    return picked

AVAILABLE_RUNS = list_story_b_runs(OUTPUTS_ROOT)
if STORY_B_RUN_FOLDERS is None or STORY_B_RUN_FOLDERS == "ALL":
    RUN_FOLDERS = AVAILABLE_RUNS
elif STORY_B_RUN_FOLDERS == "LATEST_PER_PRESET":
    RUN_FOLDERS = pick_latest_per_preset(AVAILABLE_RUNS)
else:
    # user-specified subset
    RUN_FOLDERS = list(STORY_B_RUN_FOLDERS)
    missing = [r for r in RUN_FOLDERS if r not in AVAILABLE_RUNS]
    if missing:
        print("Warning: requested runs not found (or missing association_rules.parquet):", missing)
    RUN_FOLDERS = [r for r in RUN_FOLDERS if r in AVAILABLE_RUNS]

if not RUN_FOLDERS:
    raise RuntimeError(
        "No Story B.1 runs found under data/story-module-outputs/. "
        "Expected folders like story_b/preset_.../<RUN_ID>/tables/association_rules.parquet."
    )

ACTIVE_RUN = ACTIVE_RUN or RUN_FOLDERS[0]

def run_paths(run_folder: str) -> dict[str, Path]:
    root = OUTPUTS_ROOT / run_folder
    return {
        "run_folder": Path(run_folder),
        "root": root,
        "tables": root / "tables",
        "reports": root / "reports",
        "figures": root / "figures",
    }

print("REPO_ROOT     :", REPO_ROOT)
print("TABLE_DIR     :", TABLE_DIR)
print("OUTPUTS_ROOT  :", OUTPUTS_ROOT)
print("STORY_B_ROOT  :", STORY_B_ROOT)
print("AVAILABLE_RUNS:", AVAILABLE_RUNS)
print("RUN_FOLDERS   :", RUN_FOLDERS)
print("ACTIVE_RUN    :", ACTIVE_RUN)

In [ ]:
# --- Load Story B.1 artifacts for ALL runs ---
run_data: dict[str, dict[str, Any]] = {}

print(f"[Eval] Loading Story B.1 artifacts for {len(RUN_FOLDERS)} run(s)...")
for run_folder in RUN_FOLDERS:
    print(f"[Eval] -> Loading run: {run_folder}")
    paths = run_paths(run_folder)
    tables = paths["tables"]
    reports = paths["reports"]

    manifest_path = reports / "run_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else None

    rules_path = tables / "association_rules.parquet"
    hr_path = tables / "rules_human_readable.parquet"
    itemsets_path = tables / "frequent_itemsets.parquet"

    rules_df = pd.read_parquet(rules_path)
    rules_hr_df = pd.read_parquet(hr_path) if hr_path.exists() else None
    itemsets_df = pd.read_parquet(itemsets_path) if itemsets_path.exists() else None

    run_data[run_folder] = {
        **paths,
        "manifest": manifest,
        "rules": rules_df,
        "rules_hr": rules_hr_df,
        "itemsets": itemsets_df,
    }

    print(f"[{run_folder}] rules rows: {len(rules_df):,}")
    if rules_hr_df is not None:
        print(f"[{run_folder}] rules_hr rows: {len(rules_hr_df):,}")
    if itemsets_df is not None:
        print(f"[{run_folder}] itemsets rows: {len(itemsets_df):,}")

print("[Eval] Done loading artifacts.")

# If ACTIVE_RUN wasn't explicitly set and the first run is empty, switch to first non-empty run for plots.
if not ACTIVE_RUN_WAS_SET:
    if len(run_data[ACTIVE_RUN]["rules"]) == 0:
        non_empty = [rf for rf in RUN_FOLDERS if len(run_data[rf]["rules"]) > 0]
        if non_empty:
            prev = ACTIVE_RUN
            ACTIVE_RUN = non_empty[0]
            print(f"[Eval] ACTIVE_RUN auto-switched (empty rules): {prev} -> {ACTIVE_RUN}")
        else:
            print("[Eval] Warning: all runs have 0 rules; plots will be empty.")

# Convenience: bind active run artifacts for the downstream (interpretation/plots) cells
active = run_data[ACTIVE_RUN]
rules = active["rules"]
rules_hr = active["rules_hr"]
itemsets = active["itemsets"]
manifest = active["manifest"]
RUN_ROOT = active["root"]
RUN_TABLES = active["tables"]
RUN_REPORTS = active["reports"]
RUN_FIGURES = active["figures"]

print("Active run root:", RUN_ROOT)
print("manifest found:", bool(manifest))
print("rules cols:", list(rules.columns))
display(rules.head(5))

if rules_hr is not None:
    print("rules_hr cols:", list(rules_hr.columns))
    display(rules_hr.head(5))

if itemsets is not None:
    print("itemsets cols:", list(itemsets.columns))
    display(itemsets.head(5))

In [ ]:
# --- Robust rule typing + token-family summaries (for ALL runs) ---
def _iter_tokens(obj) -> list[str]:
    if obj is None:
        return []
    # pandas NA handling
    if isinstance(obj, float) and pd.isna(obj):
        return []
    if isinstance(obj, str):
        return [obj]
    if isinstance(obj, np.ndarray):
        flat = obj.tolist()
        if isinstance(flat, str):
            return [flat]
        if isinstance(flat, list):
            return [t for t in flat if isinstance(t, str)]
        return []
    if isinstance(obj, (set, frozenset, list, tuple)):
        return [t for t in obj if isinstance(t, str)]
    return []

def _families_from_tokens(tokens_obj) -> set[str]:
    tokens = _iter_tokens(tokens_obj)
    return {t.split(":", 1)[0] for t in tokens if ":" in t}

def _label(families: set[str]) -> str:
    if families == {"genre"}:
        return "genre-only"
    if families == {"movie"}:
        return "movie-only"
    if families == {"tag"}:
        return "tag-only"
    return "cross-type"

def _col_as_series(df: pd.DataFrame, col: str) -> pd.Series:
    s = df[col]
    if isinstance(s, pd.DataFrame):
        return s.iloc[:, 0]
    return s

def _pick_cols(df: pd.DataFrame) -> tuple[str, str]:
    # Support both schemas: antecedents/consequents (plural) and antecedent/consequent (singular)
    if "antecedents" in df.columns and "consequents" in df.columns:
        return "antecedents", "consequents"
    if "antecedent" in df.columns and "consequent" in df.columns:
        return "antecedent", "consequent"
    raise KeyError("Rules dataframe missing antecedent(s)/consequent(s) columns")

def add_rule_type(rules_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    """Return (typed_rules, ante_series, cons_series).
    Also normalizes columns to df['antecedents'] and df['consequents'] for downstream cells."""
    df = rules_df.copy()
    ante_col, cons_col = _pick_cols(df)
    ante = _col_as_series(df, ante_col)
    cons = _col_as_series(df, cons_col)

    fam = ante.map(_families_from_tokens).combine(
        cons.map(_families_from_tokens),
        lambda a, b: (a or set()) | (b or set()),
    )
    df["rule_type"] = fam.map(_label)
    # normalized names for downstream cells
    df["antecedents"] = ante
    df["consequents"] = cons
    return df, df["antecedents"], df["consequents"]

summ_rows: list[dict[str, Any]] = []
print(f"[Eval] Adding rule_type for {len(run_data)} run(s)...")
for run_folder, ctx in run_data.items():
    print(f"[Eval] -> Typing rules: {run_folder}")
    typed, ante_s, cons_s = add_rule_type(ctx["rules"])
    run_data[run_folder]["rules_typed"] = typed
    run_data[run_folder]["ante_series"] = ante_s
    run_data[run_folder]["cons_series"] = cons_s

    vc = typed["rule_type"].value_counts()
    summ_rows.append({
        "run_folder": run_folder,
        "n_rules": int(len(typed)),
        "n_movie_only": int(vc.get("movie-only", 0)),
        "n_genre_only": int(vc.get("genre-only", 0)),
        "n_tag_only": int(vc.get("tag-only", 0)),
        "n_cross_type": int(vc.get("cross-type", 0)),
        "n_rule_types": int(typed["rule_type"].nunique()),
    })

summary = pd.DataFrame(summ_rows).sort_values("run_folder")
print("[Eval] Done typing rules.")
display(summary)

# Bind ACTIVE_RUN typed objects for later cells
rules = run_data[ACTIVE_RUN]["rules_typed"]
ante = run_data[ACTIVE_RUN]["ante_series"]
cons = run_data[ACTIVE_RUN]["cons_series"]

print(f"Active run ({ACTIVE_RUN}) rule_type breakdown:")
display(rules["rule_type"].value_counts().to_frame("count"))

In [ ]:
# --- Metric diagnostics (ALL runs summary + ACTIVE_RUN plots) ---
SUMMARY_METRICS = ["support", "confidence", "lift"]
metric_summary_rows: list[dict[str, Any]] = []
print(f"[Eval] Metric summaries for {len(run_data)} run(s)...")
for run_folder, ctx in run_data.items():
    print(f"[Eval] -> Metrics summary: {run_folder}")
    df = ctx.get("rules_typed") if ctx.get("rules_typed") is not None else ctx["rules"]
    row: dict[str, Any] = {"run_folder": run_folder, "n_rules": int(len(df))}
    for col in SUMMARY_METRICS:
        if col not in df.columns:
            continue
        vals = pd.to_numeric(df[col], errors="coerce").dropna()
        if vals.empty:
            continue
        row[f"{col}_p50"] = float(vals.median())
        row[f"{col}_p90"] = float(vals.quantile(0.9))
        row[f"{col}_max"] = float(vals.max())
    metric_summary_rows.append(row)

metric_summary = pd.DataFrame(metric_summary_rows).sort_values("run_folder")
print("[Eval] Done metric summaries.")
display(metric_summary)

# --- Deep-dive distributions for ACTIVE_RUN only ---
metric_cols = [c for c in ["support", "confidence", "lift", "leverage", "conviction"] if c in rules.columns]
print(f"ACTIVE_RUN={ACTIVE_RUN} metric columns:", metric_cols)

if metric_cols:
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(4 * max(1, len(metric_cols)), 3))
    if len(metric_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, metric_cols):
        vals = pd.to_numeric(rules[col], errors="coerce").dropna()
        ax.hist(vals, bins=50)
        ax.set_title(col)
    plt.tight_layout()
    plt.show()

if {"support", "lift"}.issubset(rules.columns):
    fig, ax = plt.subplots(figsize=(8, 5))
    for rtype, grp in rules.groupby("rule_type"):
        ax.scatter(grp["support"], grp["lift"], s=12, alpha=0.35, label=rtype)
    ax.set_xlabel("support")
    ax.set_ylabel("lift")
    ax.set_title(f"Support vs Lift ({ACTIVE_RUN}, colored by rule_type)")
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

# Top rules table (interpretation starter, ACTIVE_RUN)
sort_key = "lift" if "lift" in rules.columns else ("confidence" if "confidence" in rules.columns else None)
if sort_key:
    cols = ["antecedents", "consequents", "support", "confidence", "lift", "rule_type"]
    cols = [c for c in cols if c in rules.columns]
    top = rules.sort_values(sort_key, ascending=False).head(20)[cols]
    display(top)

In [ ]:
# --- Popularity bias diagnostics via token_stats (train basket-frequency) ---
token_stats_path = TABLE_DIR / "token_stats.parquet"
token_stats = pd.read_parquet(token_stats_path) if token_stats_path.exists() else None
print("token_stats found:", token_stats is not None)

def _first_token(tokens) -> str | None:
    if tokens is None:
        return None
    # most common container types in our saved parquet artifacts
    if isinstance(tokens, np.ndarray):
        if tokens.size == 0:
            return None
        v = tokens.flat[0]
        return v if isinstance(v, str) else str(v)
    if isinstance(tokens, (list, tuple, set, frozenset)):
        v = next(iter(tokens), None)
        return v if isinstance(v, str) else (None if v is None else str(v))
    if isinstance(tokens, str):
        return tokens
    return None

def _col_as_series(df: pd.DataFrame, col: str) -> pd.Series:
    s = df[col]
    if isinstance(s, pd.DataFrame):
        return s.iloc[:, 0]
    return s

def _pick_cons_series(ctx: dict[str, Any], df: pd.DataFrame) -> pd.Series:
    cons_s = ctx.get("cons_series")
    if cons_s is not None:
        return cons_s
    if "consequents" in df.columns:
        return _col_as_series(df, "consequents")
    if "consequent" in df.columns:
        return _col_as_series(df, "consequent")
    raise KeyError("Missing consequent(s) column")

def _n_train_from_manifest(m: dict[str, Any] | None, fallback: int) -> int:
    if m and isinstance(m.get("n_baskets_train"), (int, float)) and m.get("n_baskets_train"):
        return int(m["n_baskets_train"])
    return int(fallback)

pop_rows: list[dict[str, Any]] = []
if token_stats is not None:
    df_tok_base = token_stats[["token", "family", "df_after"]].copy()
    fallback_n_train = int(token_stats["df_after"].max()) if len(token_stats) else 1

    print(f"[Eval] Popularity-bias summary for {len(run_data)} run(s)...")
    for run_folder, ctx in run_data.items():
        print(f"[Eval] -> Popularity stats: {run_folder}")
        df = ctx.get("rules_typed") if ctx.get("rules_typed") is not None else ctx["rules"]
        cons_s = _pick_cons_series(ctx, df)
        cons_tok = cons_s.map(_first_token)
        n_train = _n_train_from_manifest(ctx.get("manifest"), fallback=fallback_n_train)

        df_tok = df_tok_base.copy()
        df_tok["consequent_support_train"] = df_tok["df_after"] / max(1, n_train)

        tmp = pd.DataFrame({"consequent_token": cons_tok})
        if "lift" in df.columns:
            tmp["lift"] = pd.to_numeric(df["lift"], errors="coerce")
        tmp = tmp.merge(
            df_tok[["token", "family", "consequent_support_train"]],
            left_on="consequent_token",
            right_on="token",
            how="left",
        )
        pairs = tmp[["consequent_support_train", "lift"]].dropna() if "lift" in tmp.columns else pd.DataFrame()
        if "lift" in tmp.columns and len(df) and pairs.empty:
            n_missing = int(tmp["consequent_support_train"].isna().sum())
            print(f"[Eval]    note: 0 matched (token_stats) consequents; missing support for {n_missing:,}/{len(tmp):,}")
        spearman = None
        if not pairs.empty:
            spearman = float(pairs["consequent_support_train"].corr(pairs["lift"], method="spearman"))
        pop_rows.append({
            "run_folder": run_folder,
            "n_rules": int(len(df)),
            "n_pairs": int(len(pairs)),
            "n_train_used": int(n_train),
            "spearman(lift, consequent_popularity)": spearman,
        })

    pop_summary = pd.DataFrame(pop_rows).sort_values("run_folder")
    print("[Eval] Done popularity-bias summary.")
    display(pop_summary)

    # Deep-dive scatter for ACTIVE_RUN
    active_ctx = run_data[ACTIVE_RUN]
    active_df = active_ctx.get("rules_typed") if active_ctx.get("rules_typed") is not None else active_ctx["rules"]
    active_cons = _pick_cons_series(active_ctx, active_df)
    active_cons_tok = active_cons.map(_first_token)
    n_train = _n_train_from_manifest(active_ctx.get("manifest"), fallback=fallback_n_train)

    df_tok = df_tok_base.copy()
    df_tok["consequent_support_train"] = df_tok["df_after"] / max(1, n_train)

    tmp = pd.DataFrame({"consequent_token": active_cons_tok})
    if "rule_type" in active_df.columns:
        tmp["rule_type"] = active_df["rule_type"].astype(str)
    if "support" in active_df.columns:
        tmp["support"] = pd.to_numeric(active_df["support"], errors="coerce")
    if "confidence" in active_df.columns:
        tmp["confidence"] = pd.to_numeric(active_df["confidence"], errors="coerce")
    if "lift" in active_df.columns:
        tmp["lift"] = pd.to_numeric(active_df["lift"], errors="coerce")
    tmp = tmp.merge(
        df_tok[["token", "family", "consequent_support_train"]],
        left_on="consequent_token",
        right_on="token",
        how="left",
    ).rename(columns={"family": "consequent_family"})

    display(tmp[[c for c in ["rule_type", "consequent_family", "consequent_support_train", "support", "confidence", "lift"] if c in tmp.columns]].head(10))

    if "lift" in tmp.columns and "consequent_support_train" in tmp.columns:
        plot_df = tmp.dropna(subset=["consequent_support_train", "lift"])
        if plot_df.empty:
            print(f"[Eval] Note: no matched (popularity, lift) pairs to plot for ACTIVE_RUN={ACTIVE_RUN}.")
        else:
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.scatter(plot_df["consequent_support_train"], plot_df["lift"], s=12, alpha=0.25)
            ax.set_xscale("log")
            ax.set_xlabel("consequent support in train (log scale)")
            ax.set_ylabel("lift")
            ax.set_title(f"Lift vs consequent popularity (train) — {ACTIVE_RUN}")
            plt.tight_layout()
            plt.show()

In [ ]:
# --- Interpretation helpers: decode + browse neighborhoods ---
movies = pd.read_parquet(TABLE_DIR / "dim_movies_clean.parquet", columns=["movieId", "title"])
movie_title = dict(zip(movies["movieId"].astype(int), movies["title"].astype(str)))

def _iter_tokens_any(tokens) -> Iterable[str]:
    if tokens is None:
        return []
    if isinstance(tokens, np.ndarray):
        return [str(x) for x in tokens.tolist()]
    if isinstance(tokens, (set, frozenset, list, tuple)):
        return [str(x) for x in tokens]
    if isinstance(tokens, str):
        return [tokens]
    return [str(tokens)]

def decode_token(token: str) -> str:
    """Human-friendly display for a Story-B token."""
    if not isinstance(token, str) or ":" not in token:
        return str(token)
    fam, val = token.split(":", 1)
    if fam == "movie":
        try:
            mid = int(val)
            title = movie_title.get(mid)
            if title:
                return f"{title} (movie:{mid})"
            return f"movie:{mid}"
        except Exception:
            return token
    if fam == "genre":
        return f"genre:{val}"
    if fam == "tag":
        return f"tag:{val}"
    return token

def show_top_outgoing(seed_token: str, top_k: int = 20, sort_by: str = "lift") -> pd.DataFrame:
    df = rules.copy()
    # keep rules where seed appears in antecedents
    mask = df["antecedents"].apply(lambda a: seed_token in set(_iter_tokens_any(a)))
    df = df[mask].copy()
    if sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=False)
    cols = [c for c in ["antecedents", "consequents", "support", "confidence", "lift", "rule_type"] if c in df.columns]
    df = df.head(top_k)[cols]
    # decoded convenience columns
    if "antecedents" in df.columns:
        df["antecedents_decoded"] = df["antecedents"].apply(lambda xs: ", ".join(decode_token(x) for x in _iter_tokens_any(xs)))
    if "consequents" in df.columns:
        df["consequents_decoded"] = df["consequents"].apply(lambda xs: ", ".join(decode_token(x) for x in _iter_tokens_any(xs)))
    return df

# Example (edit the seed token):
seed_token = "movie:318"
display(show_top_outgoing(seed_token, top_k=15, sort_by="lift"))

In [ ]:
# --- Held-out recommendation evaluation (movie-only runs) ---
# Evaluates each run where rules are movie-only.
# Uses train likes as seeds and checks whether test likes appear in Top-K.

MAX_SEEDS_PER_USER = 20
TOP_PER_SEED = 50
max_k = max(K_LIST) if K_LIST else 10

def _single_token(xs) -> str | None:
    if xs is None:
        return None
    if isinstance(xs, np.ndarray):
        if xs.size != 1:
            return None
        v = xs.flat[0]
        return v if isinstance(v, str) else str(v)
    if isinstance(xs, (set, frozenset, list, tuple)) and len(xs) == 1:
        v = next(iter(xs))
        return v if isinstance(v, str) else (None if v is None else str(v))
    return None

def _token_to_movie_id(token: str | None) -> int | None:
    if not token or not isinstance(token, str) or ":" not in token:
        return None
    fam, val = token.split(":", 1)
    if fam != "movie":
        return None
    try:
        return int(val)
    except Exception:
        return None

def _pick_score_col(df: pd.DataFrame) -> str | None:
    if "lift" in df.columns:
        return "lift"
    if "confidence" in df.columns:
        return "confidence"
    return None

train = None
test = None
def _load_interactions() -> tuple[pd.DataFrame, pd.DataFrame]:
    train_path = TABLE_DIR / "interactions_train.parquet"
    test_path = TABLE_DIR / "interactions_test.parquet"
    if not train_path.exists() or not test_path.exists():
        raise RuntimeError("Missing interactions_{train,test}.parquet under data/preprocessed-data/tables/")
    tr = pd.read_parquet(train_path)
    te = pd.read_parquet(test_path)
    rating_col = "rating" if "rating" in tr.columns else None
    if rating_col is None:
        raise RuntimeError("Expected a rating column in interactions_* (e.g., rating).")
    return tr, te

print(f"[Eval] Held-out evaluation across {len(run_data)} run(s) (movie-only only)...")
all_eval_rows: list[dict[str, Any]] = []
for run_folder, ctx in run_data.items():
    print(f"[Eval] -> Held-out eval: {run_folder}")
    df = ctx.get("rules_typed") if ctx.get("rules_typed") is not None else ctx["rules"]
    ante_s = ctx.get("ante_series")
    cons_s = ctx.get("cons_series")
    if ante_s is None or cons_s is None:
        # fallback: handle duplicated columns defensively
        ante_s = df["antecedents"]
        cons_s = df["consequents"]
        if isinstance(ante_s, pd.DataFrame):
            ante_s = ante_s.iloc[:, 0]
        if isinstance(cons_s, pd.DataFrame):
            cons_s = cons_s.iloc[:, 0]
    score_col = _pick_score_col(df)
    is_movie_only_run = (df.get("rule_type") is not None) and (df["rule_type"].nunique() == 1) and (df["rule_type"].iloc[0] == "movie-only")

    # Build movie->movie edge table from rules (single-antecedent, single-consequent movie tokens).
    ante_tok = ante_s.map(_single_token)
    cons_tok = cons_s.map(_single_token)
    ante_mid = ante_tok.map(_token_to_movie_id)
    cons_mid = cons_tok.map(_token_to_movie_id)

    edges = pd.DataFrame({"ante_mid": ante_mid, "cons_mid": cons_mid})
    if score_col:
        edges[score_col] = pd.to_numeric(df[score_col], errors="coerce")
    edges = edges.dropna(subset=["ante_mid", "cons_mid"])
    edges["ante_mid"] = edges["ante_mid"].astype(int)
    edges["cons_mid"] = edges["cons_mid"].astype(int)
    if score_col:
        edges = edges.dropna(subset=[score_col])

    eval_metrics = None
    skip_reason = None
    if not is_movie_only_run:
        skip_reason = "not a movie-only run"
    elif edges.empty:
        skip_reason = "no movie->movie edges"
    elif score_col is None:
        skip_reason = "no score column (lift/confidence)"

    if skip_reason is not None:
        print(f"[Eval]    skipping: {skip_reason}")
        all_eval_rows.append({
            "run_folder": run_folder,
            "movie_only": bool(is_movie_only_run),
            "edges": int(len(edges)),
            "score_col": score_col,
            "skipped": True,
            "skip_reason": skip_reason,
        })
    else:
        print(f"[Eval]    movie-only run; edges={len(edges):,}; scoring={score_col}")
        # Load interactions once (shared across runs)
        if train is None or test is None:
            print("[Eval]    loading interactions_train/test... (once)")
            train, test = _load_interactions()

        edges_sorted = edges.sort_values(["ante_mid", score_col], ascending=[True, False])
        rec_lists = edges_sorted.groupby("ante_mid")["cons_mid"].apply(list).to_dict()
        score_lists = edges_sorted.groupby("ante_mid")[score_col].apply(list).to_dict()

        rating_col = "rating"
        train_like = train[train[rating_col] >= LIKE_RATING_THRESHOLD][["userId", "movieId"]].copy()
        test_like = test[test[rating_col] >= LIKE_RATING_THRESHOLD][["userId", "movieId"]].copy()
        train_like["movieId"] = train_like["movieId"].astype(int)
        test_like["movieId"] = test_like["movieId"].astype(int)

        users = np.intersect1d(train_like["userId"].unique(), test_like["userId"].unique())
        if len(users) == 0:
            skip_reason = "no users with both train likes and test likes"
            print(f"[Eval]    skipping: {skip_reason}")
            all_eval_rows.append({
                "run_folder": run_folder,
                "movie_only": True,
                "edges": int(len(edges)),
                "score_col": score_col,
                "skipped": True,
                "skip_reason": skip_reason,
            })
        else:
            if len(users) > MAX_EVAL_USERS:
                users = rng.choice(users, size=MAX_EVAL_USERS, replace=False)

            train_by_user = train_like[train_like["userId"].isin(users)].groupby("userId")["movieId"].apply(list).to_dict()
            test_by_user = test_like[test_like["userId"].isin(users)].groupby("userId")["movieId"].apply(set).to_dict()

            hits_at_k = {k: 0 for k in K_LIST}
            mrr_sum = 0.0
            n_eval = 0
            n_covered = 0

            for uid in users:
                seeds = train_by_user.get(uid, [])
                targets = test_by_user.get(uid, set())
                if not seeds or not targets:
                    continue
                if len(seeds) > MAX_SEEDS_PER_USER:
                    seeds = list(rng.choice(seeds, size=MAX_SEEDS_PER_USER, replace=False))
                seed_set = set(seeds)

                cand_score: dict[int, float] = {}
                for s in seeds:
                    recs = rec_lists.get(int(s))
                    if not recs:
                        continue
                    scs = score_lists.get(int(s)) or []
                    for j, c in enumerate(recs[:TOP_PER_SEED]):
                        if c in seed_set:
                            continue
                        score = float(scs[j]) if j < len(scs) else 0.0
                        prev = cand_score.get(c)
                        if prev is None or score > prev:
                            cand_score[c] = score

                n_eval += 1
                if not cand_score:
                    continue
                n_covered += 1

                ranked = sorted(cand_score.items(), key=lambda kv: kv[1], reverse=True)
                recs_ranked = [m for m, _ in ranked]

                for k in K_LIST:
                    topk = recs_ranked[:k]
                    if any(m in targets for m in topk):
                        hits_at_k[k] += 1

                rr = 0.0
                for rank, mid in enumerate(recs_ranked[:max_k], start=1):
                    if mid in targets:
                        rr = 1.0 / rank
                        break
                mrr_sum += rr

            eval_metrics = {
                "run_folder": run_folder,
                "score_col": score_col,
                "like_rating_threshold": float(LIKE_RATING_THRESHOLD),
                "max_eval_users": int(MAX_EVAL_USERS),
                "max_seeds_per_user": int(MAX_SEEDS_PER_USER),
                "top_per_seed": int(TOP_PER_SEED),
                "n_eval_users": int(n_eval),
                "n_covered_users": int(n_covered),
                "coverage": float(n_covered / max(1, n_eval)),
                "hit_rate_at_k": {str(k): float(hits_at_k[k] / max(1, n_eval)) for k in K_LIST},
                f"mrr@{max_k}": float(mrr_sum / max(1, n_eval)),
            }
            ctx["eval_metrics"] = eval_metrics

            flat = {
                "run_folder": run_folder,
                "movie_only": True,
                "edges": int(len(edges)),
                "score_col": score_col,
                "skipped": False,
                "coverage": eval_metrics["coverage"],
                f"mrr@{max_k}": eval_metrics[f"mrr@{max_k}"],
            }
            for k in K_LIST:
                flat[f"hit@{k}"] = eval_metrics["hit_rate_at_k"][str(k)]
            all_eval_rows.append(flat)

            print(
                f"[Eval]    done: coverage={eval_metrics['coverage']:.3f}; "
                + f"mrr@{max_k}={eval_metrics[f'mrr@{max_k}']:.3f}"
)

    # Optionally write eval artifacts into THIS run's reports folder
    if WRITE_EVAL_ARTIFACTS:
        reports_dir = ctx["reports"]
        reports_dir.mkdir(parents=True, exist_ok=True)
        out_json = reports_dir / "eval_metrics.json"
        payload = {"eval_metrics": eval_metrics, "manifest": ctx.get("manifest")}
        out_json.write_text(json.dumps(payload, indent=2), encoding="utf-8")

        out_md = reports_dir / "eval_summary.md"
        lines: list[str] = []
        lines.append(f"# Story B.1 Evaluation Summary ({run_folder})")
        lines.append("")
        if eval_metrics is None:
            lines.append(f"Held-out evaluation was skipped: {skip_reason}.")
        else:
            lines.append("## Held-out metrics")
            lines.append("")
            lines.append(f"- coverage: {eval_metrics['coverage']:.3f}")
            for k, v in eval_metrics["hit_rate_at_k"].items():
                lines.append(f"- hit_rate@{k}: {v:.3f}")
            lines.append(f"- mrr@{max_k}: {eval_metrics[f'mrr@{max_k}']:.3f}")
        lines.append("")
        m = ctx.get("manifest")
        if m is not None:
            lines.append("## Run manifest (excerpt)")
            lines.append("")
            for key in [
                "RUN_PRESET",
                "TOKEN_FAMILY_MODE",
                "ITEMSET_MINER",
                "MIN_SUPPORT_LOW",
                "MIN_CONFIDENCE",
                "MIN_LIFT",
                "MAX_ITEMSET_LEN",
            ]:
                if key in m:
                    lines.append(f"- {key}: {m[key]}")
        out_md.write_text("\n".join(lines) + "\n", encoding="utf-8")
        print(f"[Eval]    wrote reports to: {reports_dir}")

print("[Eval] Held-out evaluation finished.")
eval_summary = pd.DataFrame(all_eval_rows).sort_values("run_folder")
display(eval_summary)